# Tierkreis checkpoints; using pre build graphs

In this example we will run a predefined graph in a customized environment.

By default tierkreis will store the checkpoints of workflow runs in the filesystem at `~/.tierkreis/checkpoints`.
This can be configured in the storage by providing the `tierkreis_directory` argument to the storage layer (e.g. `checkpoints2`).


In [ ]:
from pathlib import Path
from uuid import UUID

from tierkreis.storage import FileStorage

storage = FileStorage(
    UUID(int=107),
    do_cleanup=True,
    tierkreis_directory=Path.home() / ".tierkreis" / "checkpoints2",
)

This change needs to be forwarded to the executors, which is why they take the logs path as additional argument.


In [ ]:
from tierkreis.consts import PACKAGE_PATH
from tierkreis.executor import UvExecutor

executor = UvExecutor(PACKAGE_PATH.parent / "tierkreis_workers", storage.logs_path)

Typically now one would define a graph.
Although, tierkreis ships some preconstructed graphs.
For this example we are going a to run a circuit on nexus and poll for its results.

In [ ]:
from tierkreis.graphs.nexus.submit_poll import nexus_submit_and_poll

graph = nexus_submit_and_poll()

Before we run the graph we can visualize it to get familiar with it.
This will start a webserver and listen on [localhost](http://localhost:8000)

In [ ]:
from qnexus.client.auth import login
from tierkreis_visualization.visualize_graph import visualize_graph

login()
visualize_graph(
    graph,
)  # this spawns a server, you need to manually terminate this cell.

and finally we can run it

In [ ]:
from pytket.qasm.qasm import circuit_from_qasm
from qnexus import AerConfig
from tierkreis.controller import run_graph
from tierkreis.storage import read_outputs

aer_config = AerConfig()
circuit = circuit_from_qasm(Path().parent / "data" / "ghz_state_n23.qasm")
circuits = [circuit]
inputs = {
    "project_name": "2025-tkr-test",
    "job_name": "job-1",
    "circuits": circuits,
    "n_shots": [30] * len(circuits),
    "backend_config": aer_config,
}
storage.clean_graph_files()
run_graph(
    storage,
    executor,
    graph,
    inputs,
    polling_interval_seconds=0.1,
)
res = read_outputs(graph, storage)